# ResQAI — Emergency Triage Agent (Gemini version)Free, no-credit-card version using Google's Gemini API.**Run every cell top to bottom, in order, with Shift+Enter.**

## 1. Install

In [32]:
!pip install -q google-generativeai

In [33]:
!pip install --upgrade google-generativeai

## 2. Setup — paste your Gemini API keyGet a free key at https://aistudio.google.com/apikey (no card needed). Paste it directly below.

In [34]:
import google.generativeai as genai
import json, math, time

GEMINI_API_KEY = "AQ.Ab8RN6LnmH13W5OJXedf6LIRqNiGzIfHXEsfQb0NXxelvKyvtg"

genai.configure(api_key=GEMINI_API_KEY)
print("Configured. Key starts with:", GEMINI_API_KEY[:6])

Configured. Key starts with: AQ.Ab8


## 3. Upload data filesUpload `hospitals.json` and `patients.json` using the folder icon on the left sidebar (upload directly, no subfolder needed).

In [35]:
with open("hospitals.json") as f:
    HOSPITALS = json.load(f)

with open("patients.json") as f:
    PATIENTS = json.load(f)

print(f"Loaded {len(HOSPITALS)} hospitals and {len(PATIENTS)} incoming patients")
HOSPITALS

Loaded 4 hospitals and 4 incoming patients


[{'id': 'H1',
  'name': 'City General Hospital',
  'lat': 19.9975,
  'lon': 73.7898,
  'beds': {'ICU': 3, 'ER': 8, 'General': 20}},
 {'id': 'H2',
  'name': "St. Mary's Medical Center",
  'lat': 19.9615,
  'lon': 73.7645,
  'beds': {'ICU': 1, 'ER': 5, 'General': 15}},
 {'id': 'H3',
  'name': 'Riverside Trauma Center',
  'lat': 20.011,
  'lon': 73.748,
  'beds': {'ICU': 5, 'ER': 10, 'General': 25}},
 {'id': 'H4',
  'name': 'Nashik District Hospital',
  'lat': 19.933,
  'lon': 73.81,
  'beds': {'ICU': 0, 'ER': 3, 'General': 12}}]

## 4. Tool functions (the agent calls these)

In [36]:
def calculate_news2_score(resp_rate: float, spo2: float, systolic_bp: float, hr: float, gcs: float, temp_c: float) -> dict:
    """Compute a deterministic NEWS2-style clinical risk score (0-20) from patient vitals.
    Always call this first before deciding priority tier.

    Args:
        resp_rate: respiratory rate (breaths per minute)
        spo2: oxygen saturation percentage
        systolic_bp: systolic blood pressure
        hr: heart rate (beats per minute)
        gcs: Glasgow Coma Scale score (3-15)
        temp_c: body temperature in Celsius

    Returns:
        dict with 'score' (0-20) and 'tier' (CRITICAL, URGENT, or STANDARD)
    """
    score = 0

    if resp_rate <= 8 or resp_rate >= 25:
        score += 3
    elif resp_rate >= 21:
        score += 2
    elif resp_rate <= 11:
        score += 1

    if spo2 <= 91:
        score += 3
    elif spo2 <= 93:
        score += 2
    elif spo2 <= 95:
        score += 1

    if systolic_bp <= 90 or systolic_bp >= 220:
        score += 3
    elif systolic_bp <= 100:
        score += 2
    elif systolic_bp <= 110:
        score += 1

    if hr <= 40 or hr >= 131:
        score += 3
    elif hr >= 111:
        score += 2
    elif hr <= 50 or hr >= 91:
        score += 1

    if gcs < 15:
        score += 3

    if temp_c <= 35.0:
        score += 3
    elif temp_c >= 39.1:
        score += 2
    elif temp_c >= 38.1 or temp_c <= 36.0:
        score += 1

    if score >= 9:
        tier = "CRITICAL"
    elif score >= 5:
        tier = "URGENT"
    else:
        tier = "STANDARD"

    return {"score": score, "tier": tier}


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def find_and_reserve_bed(patient_lat: float, patient_lon: float, tier: str) -> dict:
    """Find the nearest hospital with an available bed matching the patient's priority tier, and reserve it.

    Args:
        patient_lat: patient latitude
        patient_lon: patient longitude
        tier: one of CRITICAL, URGENT, STANDARD

    Returns:
        dict describing the reservation, or NO_BED_AVAILABLE status if none found
    """
    bed_type = {"CRITICAL": "ICU", "URGENT": "ER", "STANDARD": "General"}[tier]

    candidates = []
    for h in HOSPITALS:
        available = h["beds"].get(bed_type, 0)
        if available > 0:
            dist = haversine_km(patient_lat, patient_lon, h["lat"], h["lon"])
            candidates.append((dist, h))

    if not candidates:
        return {"status": "NO_BED_AVAILABLE", "bed_type": bed_type}

    candidates.sort(key=lambda x: x[0])
    dist, hospital = candidates[0]
    hospital["beds"][bed_type] -= 1

    return {
        "status": "RESERVED",
        "hospital_id": hospital["id"],
        "hospital_name": hospital["name"],
        "bed_type": bed_type,
        "distance_km": round(dist, 2),
        "remaining_beds": hospital["beds"][bed_type],
    }


def check_deterioration_trend(vitals_history: list) -> dict:
    """Rule-based threshold on vitals trend -- simple stand-in for a predictive model."""
    if len(vitals_history) < 2:
        return {"alert": False}
    first, last = vitals_history[0], vitals_history[-1]
    spo2_drop = first.get("spo2", 100) - last.get("spo2", 100)
    hr_rise = last.get("hr", 0) - first.get("hr", 0)
    if spo2_drop >= 4 or hr_rise >= 20:
        return {"alert": True, "reason": f"SpO2 dropped {spo2_drop} pts / HR rose {hr_rise} bpm"}
    return {"alert": False}

print("Tools defined.")

Tools defined.


## 5. Build the agent (Gemini automatic function calling)

In [39]:
def run_triage_agent(patient):
    vitals = patient["vitals"]
    score_result = calculate_news2_score(
        resp_rate=vitals["resp_rate"],
        spo2=vitals["spo2"],
        systolic_bp=vitals["systolic_bp"],
        hr=vitals["hr"],
        gcs=vitals["gcs"],
        temp_c=vitals["temp_c"],
    )
    tier = score_result["tier"]

    print("[TOOL CALL] calculate_news2_score", vitals)
    print("[TOOL RESULT]", score_result)

    bed_result = find_and_reserve_bed(patient["lat"], patient["lon"], tier)

    print("[TOOL CALL] find_and_reserve_bed", {"tier": tier, "lat": patient["lat"], "lon": patient["lon"]})
    print("[TOOL RESULT]", bed_result)

    if bed_result["status"] == "RESERVED":
        summary = (
            f"Patient scored {score_result['score']}/20 -> priority tier {tier}. "
            f"Reserved a {bed_result['bed_type']} bed at {bed_result['hospital_name']} "
            f"({bed_result['distance_km']} km away). "
            f"{bed_result['remaining_beds']} {bed_result['bed_type']} beds remaining there."
        )
    else:
        summary = (
            f"Patient scored {score_result['score']}/20 -> priority tier {tier}. "
            f"NO {bed_result['bed_type']} bed available nearby -- escalating to human dispatcher."
        )

    print("[AGENT SUMMARY]")
    print(summary)
    print()

    return summary

print("Agent ready (offline, deterministic mode).")

Agent ready (offline, deterministic mode).


## 6. Run on one patient (single test)

In [40]:
result = run_triage_agent(PATIENTS[0])

[TOOL CALL] calculate_news2_score {'hr': 130, 'systolic_bp': 82, 'resp_rate': 28, 'spo2': 89, 'gcs': 13, 'temp_c': 38.9}
[TOOL RESULT] {'score': 15, 'tier': 'CRITICAL'}
[TOOL CALL] find_and_reserve_bed {'tier': 'CRITICAL', 'lat': 19.99, 'lon': 73.785}
[TOOL RESULT] {'status': 'RESERVED', 'hospital_id': 'H1', 'hospital_name': 'City General Hospital', 'bed_type': 'ICU', 'distance_km': 0.97, 'remaining_beds': 2}
[AGENT SUMMARY]
Patient scored 15/20 -> priority tier CRITICAL. Reserved a ICU bed at City General Hospital (0.97 km away). 2 ICU beds remaining there.



## 7. Full demo — all patients

In [41]:
with open("hospitals.json") as f:
    HOSPITALS = json.load(f)

for p in PATIENTS:
    print()
    print("--- Incoming:", p["name"], "(" + p["id"] + ") ---")
    print("Description:", p["description"])
    run_triage_agent(p)
    time.sleep(1)

print()
print("=== Demo complete ===")


--- Incoming: Patient A (P1) ---
Description: Motorcycle accident, visible chest trauma, labored breathing
[TOOL CALL] calculate_news2_score {'hr': 130, 'systolic_bp': 82, 'resp_rate': 28, 'spo2': 89, 'gcs': 13, 'temp_c': 38.9}
[TOOL RESULT] {'score': 15, 'tier': 'CRITICAL'}
[TOOL CALL] find_and_reserve_bed {'tier': 'CRITICAL', 'lat': 19.99, 'lon': 73.785}
[TOOL RESULT] {'status': 'RESERVED', 'hospital_id': 'H1', 'hospital_name': 'City General Hospital', 'bed_type': 'ICU', 'distance_km': 0.97, 'remaining_beds': 2}
[AGENT SUMMARY]
Patient scored 15/20 -> priority tier CRITICAL. Reserved a ICU bed at City General Hospital (0.97 km away). 2 ICU beds remaining there.


--- Incoming: Patient B (P2) ---
Description: Fall from ladder, suspected wrist fracture, alert and stable
[TOOL CALL] calculate_news2_score {'hr': 95, 'systolic_bp': 118, 'resp_rate': 18, 'spo2': 97, 'gcs': 15, 'temp_c': 37.1}
[TOOL RESULT] {'score': 1, 'tier': 'STANDARD'}
[TOOL CALL] find_and_reserve_bed {'tier': 'STANDAR

## 8. Bonus: predictive deterioration check

In [42]:
simulated_stream = [
    {"spo2": 95, "hr": 90, "t": 0},
    {"spo2": 93, "hr": 100, "t": 60},
    {"spo2": 89, "hr": 118, "t": 120},
]
print(json.dumps(check_deterioration_trend(simulated_stream), indent=2))

{
  "alert": true,
  "reason": "SpO2 dropped 6 pts / HR rose 28 bpm"
}
